In [ ]:

sample_data = [
    """FAQ: What is the HR email ID at Dummy Company Pvt. Ltd.?
You can reach the HR department at hr@dummycompany.com for any recruitment, payroll, or employee verification queries.
Response time is typically within 2–3 business days.""",

    """Office Address:
Dummy Company Pvt. Ltd.
4th Floor, Sunrise Tech Park, MG Road, Bengaluru, India – 560001.
Working hours: Monday to Friday, 9:00 AM – 6:00 PM.""",

    """Customer Support Contact:
For product issues or warranty claims, contact support@dummycompany.com.
Toll-free helpline: 1800-202-4455 (available 9 AM to 8 PM IST).""",

    """About Us:
Dummy Company Pvt. Ltd. is a software solutions provider specializing in AI and automation products.
Founded in 2014, we serve clients across finance, healthcare, and e-commerce sectors.""",

    """Employee Benefits:
We offer medical insurance, paid leaves, and annual bonuses.
Employees can log in to the HR portal (https://portal.dummycompany.com) for claim details.""",

    """Careers:
We’re always looking for passionate developers and analysts!
To apply, visit our Careers page or send your resume to careers@dummycompany.com.""",

    """Vendor Payments:
All vendor invoices must be submitted by the 5th of each month to accounts@dummycompany.com.
Payments are processed by the 15th of every month.""",

    """Privacy Policy:
Dummy Company Pvt. Ltd. ensures complete data confidentiality.
We never sell or share customer data with third parties without consent.""",

    """IT Support:
For technical assistance, contact IT helpdesk via ticket system (helpdesk@dummycompany.com).
Emergency support available 24/7 for server-related issues.""",

    """CEO’s Message:
“At Dummy Company, innovation and integrity are our core values.
We believe in building technology that simplifies lives.” — Rohan, CEO."""
]


In [ ]:
from persistent import initial_add_data

initial_add_data(sample_data)

In [ ]:
# after adding in VDB , only use search
from persistent import search

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = "your-api-key"

llm = init_chat_model("openai:gpt-4.1")

# RAG as TOOL

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition


@tool
def get_company_details(query: str) -> str:
    """
    Retrieves company information for DummyCorp using RAG search.
    Example: get_company_details("DummyCorp financial report")
    """
    results = search(query, 2)
    
    if not results:
        return "No relevant company details found for DummyCorp."
    
    # Combine the top 2 search results into a readable response
    combined = "\n\n".join(results)
    return f"Here are the top details about DummyCorp:\n\n{combined}"
    
# Augment the LLM with tools
tools = [get_company_details]
tools_by_name = {tool.name: tool for tool in tools}
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate
from langgraph.checkpoint.memory import MemorySaver
from typing import Literal
from langchain_core.messages import ToolMessage

class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)


customerSupport_template = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly customer support assistant for DummyCorp. Give clear, polite, and helpful replies in maximum 20 words."),
    ("human", "{input}")
])

customerSupport_chain = customerSupport_template | llm_with_tools

# Agent
def customerSupportAgent(state: State):
    # print(f"Before State : {state} \n\n")
    response = customerSupport_chain.invoke(state["messages"])
    # print(f"After State : {state} \n\n")
    return {"messages": [response]}


def tool_node(state: dict):
    """Performs the tool call"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}


def should_continue(state: State) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]
    # If the LLM makes a tool call, then perform an action
    if last_message.tool_calls:
        return "tool_node"
    # Otherwise, we stop (reply to the user)
    return END

    
graph_builder.add_node("customerSupport", customerSupportAgent)
graph_builder.add_node("tool_node", tool_node)

graph_builder.add_edge(START, "customerSupport")
graph_builder.add_conditional_edges(
    "customerSupport",
    should_continue,
    ["tool_node", END]
)
graph_builder.add_edge("tool_node", "customerSupport")

graph_builder.add_edge("customerSupport", END)

# Compile with memory
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

# Config with thread ID for chat history
config = {"configurable": {"thread_id": "my_conversation"}}


In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
def stream_graph_updates(user_input: str):
    result = graph.invoke(
        {"messages": [{"role": "user", "content": user_input}]},config=config)
    if result["messages"]:
        print("Assistant:", result["messages"][-1].content)

while True:
    try:
        user_input = input("\nUser: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates(user_input)
    except:
        user_input = "What do you know about LangGraph?"
        print("User: " + user_input)
        stream_graph_updates(user_input)
        break

# RAG as Agent

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)


customerSupport_template = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly customer support assistant for DummyCorp. Give clear, polite, and helpful replies in maximum 20 words."),
    ("human", "{input}")
])

customerSupport_chain = customerSupport_template | llm

# Agent
def customerSupportAgent(state: State):
    print("\n\nState :", state,"\n\n")
    response = customerSupport_chain.invoke(state["messages"])
    return {"messages": [response]}

    
def searchAgent(state: State):
    last_msg = state["messages"][-1]
    query = last_msg.content if hasattr(last_msg, 'content') else last_msg.get("content", "")
    
    search_results = search(query, 2)
    return {"messages": ["Context : \n"+search_results+"\n\n"]}
    
graph_builder.add_node("customerSupport", customerSupportAgent)
graph_builder.add_node("searchAgent", searchAgent)

graph_builder.add_edge(START, "searchAgent")
graph_builder.add_edge("searchAgent","customerSupport")

graph_builder.add_edge("customerSupport", END)


# Compile with memory
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

# Config with thread ID for chat history
config = {"configurable": {"thread_id": "my_conversation"}}


In [ ]:
def stream_graph_updates(user_input: str):
    result = graph.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        config=config
    )
    if result["messages"]:
        print("Assistant:", result["messages"][-1].content)

while True:
    try:
        user_input = input("\nUser: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates(user_input)
    except:
        user_input = "What do you know about LangGraph?"
        print("User: " + user_input)
        stream_graph_updates(user_input)
        break

In [ ]:
# Exatracts whatever it is , so response will be affected

# Storing Chat history in DB

In [ ]:
import os
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = "your-api-key"

llm = init_chat_model("openai:gpt-4.1")

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
conn = sqlite3.connect("checkpoints.sqlite", check_same_thread=False)
memory = SqliteSaver(conn)

class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)


customerSupport_template = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly customer support assistant for DummyCorp. Give clear, polite, and helpful replies in maximum 20 words."),
    ("human", "{input}")
])

customerSupport_chain = customerSupport_template | llm

# Agent
def customerSupportAgent(state: State):
    print("\n\nState :", state,"\n\n")
    response = customerSupport_chain.invoke(state["messages"])
    return {"messages": [response]}


graph_builder.add_node("customerSupport", customerSupportAgent)

graph_builder.add_edge(START, "customerSupport")
graph_builder.add_edge("customerSupport", END)


graph = graph_builder.compile(checkpointer=memory)

# Config with thread ID for chat history
config = {"configurable": {"thread_id": "my_conversation"}}


In [ ]:
def stream_graph_updates(user_input: str):
    result = graph.invoke(
        {"messages": [{"role": "user", "content": user_input}]},config=config)
    if result["messages"]:
        print("Assistant:", result["messages"][-1].content)

while True:
    try:
        user_input = input("\nUser: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates(user_input)
    except:
        user_input = "What do you know about LangGraph?"
        print("User: " + user_input)
        stream_graph_updates(user_input)
        break

# Parallel Node Execution 

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict


class State(TypedDict):
    message: str
    add_result: int
    mul_result: int
    final_result: int


graph_builder = StateGraph(State)


def addAgent(state: State):
    """Adds 10 to the input number"""
    number = state["message"]
    output = int(number) + 10
    print(f"addAgent: {number} + 10 = {output}")
    return {"add_result": output}


def mulAgent(state: State):
    """Multiplies the input number by 10"""
    number = state["message"]
    output = int(number) * 10
    print(f"mulAgent: {number} * 10 = {output}")
    return {"mul_result": output}


def combineAgent(state: State):
    """Combines results from both parallel agents"""
    combined = state["add_result"] + state["mul_result"]
    print(f"combineAgent: {state['add_result']} + {state['mul_result']} = {combined}")
    return {"final_result": combined}


# Add nodes
graph_builder.add_node("addAgent", addAgent)
graph_builder.add_node("mulAgent", mulAgent)
graph_builder.add_node("combineAgent", combineAgent)

graph_builder.add_edge(START, "addAgent")
graph_builder.add_edge(START, "mulAgent")

graph_builder.add_edge("addAgent", "combineAgent")
graph_builder.add_edge("mulAgent", "combineAgent")

graph_builder.add_edge("combineAgent", END)

graph = graph_builder.compile()
result = graph.invoke({"message": "15"})

print("\nFinal Result:")
print(result)

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict


class State(TypedDict):
    message: str
    preprocessed: int
    add_result: int
    mul_result: int
    final_result: int


graph_builder = StateGraph(State)


def preprocessAgent(state: State):
    """First agent: preprocesses the input"""
    number = state["message"]
    output = int(number) + 5
    print(f"preprocessAgent: {number} + 5 = {output}")
    return {"preprocessed": output}


def addAgent(state: State):
    """Parallel agent 1: Adds 10 to preprocessed number"""
    number = state["preprocessed"]
    output = number + 10
    print(f"addAgent: {number} + 10 = {output}")
    return {"add_result": output}


def mulAgent(state: State):
    """Parallel agent 2: Multiplies preprocessed number by 10"""
    number = state["preprocessed"]
    output = number * 10
    print(f"mulAgent: {number} * 10 = {output}")
    import time
    time.sleep(4)
    return {"mul_result": output}


def combineAgent(state: State):
    """Combines results from both parallel agents"""
    combined = state["add_result"] + state["mul_result"]
    print(f"combineAgent: {state['add_result']} + {state['mul_result']} = {combined}")
    return {"final_result": combined}


# Add nodes
graph_builder.add_node("preprocessAgent", preprocessAgent)
graph_builder.add_node("addAgent", addAgent)
graph_builder.add_node("mulAgent", mulAgent)
graph_builder.add_node("combineAgent", combineAgent)

graph_builder.add_edge(START, "preprocessAgent")

graph_builder.add_edge("preprocessAgent", "addAgent")
graph_builder.add_edge("preprocessAgent", "mulAgent")

graph_builder.add_edge("addAgent", "combineAgent")
graph_builder.add_edge("mulAgent", "combineAgent")

graph_builder.add_edge("combineAgent", END)

# Compile and run
graph = graph_builder.compile()


from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass


In [ ]:
result = graph.invoke({"message": "15"})

print("\nFinal Result:")
print(result)